In [ ]:
# ==========================================================================
# corrected per-sector F1 (v2) — replaces the previous correction
#
# definition: per-sector F1 = macro-average of per-class F1s for the classes
# belonging to that sector, where each per-class F1 is computed on the FULL
# test set (not filtered to in-sector samples).
#
# why v2: the v1 correction filtered to in-sector samples before computing
# F1, which artificially shrinks the false-positive count for in-sector
# classes (out-of-sector samples mis-predicted as in-sector classes don't
# get counted as FPs). this inflates precision and F1. the v2 definition
# matches the natural reading "average per-class performance across the
# sector's classes."
# ==========================================================================

import json
import numpy as np
from sklearn.metrics import f1_score
from pathlib import Path

SECTOR_TO_CLASS_IDX = {
    'energy':    [0, 1, 2, 3, 4, 5],
    'water':     [6, 7, 8],
    'transport': [9, 10, 11],
    'telecom':   [12],
}
N_CLASSES = 13

CLASS_NAMES = [
    'tx_substation', 'dx_substation', 'dx_other',
    'power_plant', 'solar_farm', 'wind_farm',
    'wastewater', 'water_works', 'storage_tank',
    'airport', 'train_station', 'port_terminal',
    'data_center',
]


def find_test_dict(results):
    """Locate the dict containing 'confusion' / 'per_sector' regardless of
    where the file nests it (flat / .test / .linear_probe.test)."""
    if 'confusion' in results and 'per_sector' in results:
        return results
    if 'test' in results and isinstance(results['test'], dict):
        return results['test']
    if 'linear_probe' in results and 'test' in results['linear_probe']:
        return results['linear_probe']['test']
    raise ValueError(
        f'Cannot find test result dict. Top-level keys: {list(results.keys())}'
    )


def per_sector_f1_from_confusion_v2(cm):
    """Macro-avg of per-class F1s within each sector, computed on the FULL
    test set. The natural "average performance across the sector's classes"
    definition."""
    cm = np.array(cm)
    assert cm.shape == (N_CLASSES, N_CLASSES), f'Expected {N_CLASSES}x{N_CLASSES}, got {cm.shape}'

    # reconstruct full y_true / y_pred from the whole confusion matrix
    y_true_all, y_pred_all = [], []
    for true_cls in range(N_CLASSES):
        for pred_cls in range(N_CLASSES):
            count = int(cm[true_cls, pred_cls])
            y_true_all.extend([true_cls] * count)
            y_pred_all.extend([pred_cls] * count)

    # per-class F1 across all classes (computed once)
    per_class_f1 = f1_score(
        y_true_all, y_pred_all,
        labels=list(range(N_CLASSES)),
        average=None,
        zero_division=0.0,
    )

    out = {}
    for sector, class_indices in SECTOR_TO_CLASS_IDX.items():
        sector_f1s = [float(per_class_f1[i]) for i in class_indices]
        macro_f1 = float(np.mean(sector_f1s))

        # n = number of test samples whose true label is in this sector
        n_sector = sum(int(cm[i, :].sum()) for i in class_indices)
        # diagonal hits within the sector
        correct = sum(int(cm[i, i]) for i in class_indices)
        acc = correct / n_sector if n_sector > 0 else float('nan')

        out[sector] = {
            'n': n_sector,
            'macro_f1': macro_f1,
            'acc': acc,
            'per_class_f1_in_sector': {
                CLASS_NAMES[i]: sector_f1s[idx]
                for idx, i in enumerate(class_indices)
            },
        }
    return out


# --- files to update ---
RESULTS_DIR = Path('/content/drive/MyDrive/infra_fm/results')
RESULT_FILES = [
    ('SatlasPretrain S2',
     RESULTS_DIR / 'fm_eval_satlas_multisector_v1' / 'satlas_s2_full7region_v1_linear_probe_capped_results.json'),
    ('SatlasPretrain S1',
     RESULTS_DIR / 'fm_eval_satlas_s1_v1' / 'satlas_s1_v1_best_checkpoint_test_result.json'),
    ('CROMA base',
     RESULTS_DIR / 'fm_eval_croma_v1' / 'croma_base_v1_linear_probe_results.json'),
]


for label, path in RESULT_FILES:
    print(f'\n=== {label} ===')
    print(f'Path: {path}')
    if not path.exists():
        print('  SKIP: file not found')
        continue

    with open(path) as f:
        results = json.load(f)

    try:
        test_dict = find_test_dict(results)
    except ValueError as e:
        print(f'  SKIP: {e}')
        continue

    if 'confusion' not in test_dict:
        print('  SKIP: no "confusion" key in test dict')
        continue

    cm = test_dict['confusion']
    corrected = per_sector_f1_from_confusion_v2(cm)
    original = test_dict.get('per_sector', {})
    prev_correction = test_dict.get('per_sector_corrected', {})

    print(f"  {'Sector':<12} {'n':>6} {'Original':>9} {'v1 (bad)':>9} {'v2 (good)':>10}")
    print(f"  {'-'*12} {'-'*6} {'-'*9} {'-'*9} {'-'*10}")
    for sector in ['energy', 'water', 'transport', 'telecom']:
        new = corrected[sector]
        orig_f1 = original.get(sector, {}).get('macro_f1', float('nan'))
        v1_f1 = prev_correction.get(sector, {}).get('macro_f1', float('nan'))
        v2_f1 = new['macro_f1']
        print(f"  {sector:<12} {new['n']:>6d} {orig_f1:>9.4f} {v1_f1:>9.4f} {v2_f1:>10.4f}")

    # overwrite the previous (incorrect) correction with v2
    test_dict['per_sector_corrected'] = corrected
    test_dict['per_sector_correction_note'] = (
        'per_sector_corrected = macro-average of per-class F1s for the '
        "sector's classes, where each per-class F1 is computed on the full "
        'test set. This matches the natural definition: average performance '
        "across the sector's classes. Replaces an earlier v1 correction that "
        'filtered to in-sector samples before computing F1, which incorrectly '
        'undercounted false positives for in-sector classes.'
    )

    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'  Updated: saved v2 per_sector_corrected to {path.name}')


print('\nDone.')